# GitOps: how fast is CO2 rising?

## Learning objectives

- Declare every package in `environment.yml`, and lock the environment for your platform.
- Find the commit that changed a result with `git bisect run`.
- Stop a stale result from being committed, with a pre-commit hook.
- Delete the clone and the environment, rebuild from the remote, and get the same number.

## The problem

- `project` is a Git repository that computes one number: how fast CO2 at Mauna Loa is rising now, in ppm per year.
- `python analyze.py` fails. It imports a package that `environment.yml` doesn't declare.
- Once it runs, it doesn't reproduce the number in `metrics.json`. One commit changed a parameter, and nobody reran it.
- The tests pass anyway.

## What you'll do

1. Step 1: declare the missing package, build the environment and write the lock. This is the short path, about 15 minutes, and it has its own check.
2. Step 2, optional: find the commit that moved the number, choose a degree and install a hook.
3. Step 3, optional: push, delete the clone and the environment, rebuild from the remote and tag the release.

- Steps 2 and 3 take about 15 more minutes.
- Only reading? Each step shows the failure and how to approach the fix.

## The rule

- The document and the data are the pets. Everything that runs them is cattle.
- Here the pets are what's in Git. The cattle are your clone and your conda environment.

## Get a terminal

- You read this page and paste commands into a terminal. Nothing runs on the page.
- The output shown here is recorded from the untouched project, so it stops at the failures. The fixes are yours.

**[Open it in Binder](https://mybinder.org/v2/gh/ohpcsim/reproducibility-in-ai/binder-01-gitops?urlpath=lab/workspaces/exercise)**

- Free, with nothing to install. The first start can take a few minutes.
- The terminal is on the left, already in the `project` folder. This page is on the right.
- Copy a command: select it and press Ctrl+C, or Cmd+C on a Mac.
- Paste it in the terminal: Ctrl+Shift+V, or right-click and choose Paste. Cmd+V on a Mac is untested, so right-click if it doesn't work.
- Edit a file: double-click it in the file panel on the far left. Save with Ctrl+S.
- A session is temporary and has 2 GB of memory. Download anything you want to keep.

**Or use your own terminal**

- Linux or macOS, on x86_64 or ARM, or Windows through WSL2. On Windows, keep the folder in your WSL2 home, not under `/mnt/c`.
- You need [conda](https://github.com/conda-forge/miniforge). You don't need Jupyter.
- Paste this once:

```bash
git clone --branch binder-01-gitops --single-branch https://github.com/ohpcsim/reproducibility-in-ai.git gitops
cd gitops
conda env create --quiet --file .binder/environment.yml
conda activate repro-gitops
python scripts/setup_project.py
cd project
```

- Every command below runs in that `project` folder.
- To start over, go to the folder above it and run `python scripts/setup_project.py --reset`. This deletes your work.

## The repository

- `project` has six commits by three people.
- `../remote` stands in for GitHub.

```bash
git log --format='%h  %an: %s'
```

In [1]:
# Recorded output. You don't need to run this.
import sys

sys.path.insert(0, ".exercise")
from page import run

run("git log --format='%h  %an: %s'")

$ git log --format='%h  %an: %s'
a06a448  Sam Okafor: Describe the project in the README
bbd77c3  Lee Tran: Read the CSV with pandas
805e276  Lee Tran: Tidy params.yaml
e358e01  Sam Okafor: Add a test of the fit and a holdout comparison
303c8f0  Priya Rao: Start the fit at 2000, and rerun
8c62c2e  Priya Rao: Add the Mauna Loa data and a first trend fit


## Step 1: make it run

**Problem:** the README says to run `python analyze.py`, and it fails.

```bash
python analyze.py
```

In [2]:
# Recorded output. You don't need to run this.
run("python analyze.py")

$ python analyze.py
Traceback (most recent call last):
  File "~/project/analyze.py", line 4, in <module>
    import pandas as pd
ModuleNotFoundError: No module named 'pandas'


**Facts**

- `analyze.py` imports pandas. It was installed on one laptop and never declared, so it runs there and nowhere else.
- Add a package to `environment.yml` when you start using it.
- `environment.yml` lists what you asked for. The lock lists what you got.
- Don't pin versions in `environment.yml`. That's the lock's job.
- Each platform needs its own lock, so the lock's name includes the platform, such as `linux-64` or `osx-arm64`.

**Steps**

1. Open `environment.yml` in the `project` folder. Add the missing package on a new line, in the same form as the others, with no version. Save the file.
2. Confirm the edit before you build anything. This should show the one line you added. If it prints nothing, you edited another file or didn't save.

```bash
git diff environment.yml
```

3. Build the environment from that file, write the lock and commit both. This takes a minute or two. The commit should say `2 files changed`.

```bash
conda env create --quiet --prefix ../envs/co2 --file environment.yml
conda list --prefix ../envs/co2 --explicit --md5 > explicit-$(conda config --show subdir | cut -d' ' -f2).txt
git add environment.yml explicit-*.txt
git commit -m "Declare pandas and lock the environment"
```

4. Run the analysis in that environment, look at the change to `metrics.json`, and undo it. Don't commit `metrics.json` yet. Step 2 covers why.

```bash
../envs/co2/bin/python analyze.py
git diff metrics.json
git restore metrics.json
```

### Check the short path

- Run this when step 1 is done. Each line says what to do next.
- When it passes, anyone can build your environment from what's in Git. That's the end of the short path.

```bash
python ../check.py --short
```

On the starter project it fails like this:

In [3]:
# Recorded output. You don't need to run this.
run("python ../check.py --short")

$ python ../check.py --short
FAIL  analyze.py imports pandas, and the committed environment.yml does not declare it
FAIL  No lock file is committed. Write one with conda list --explicit --md5 and commit it.
FAIL  No environment at envs/co2. Build it with the commands in step 1.

0 of 3 parts pass.


## Step 2 (optional): find out why the number moved

**Problem**

- A rerun of the committed `params.yaml` and data doesn't give the number in `metrics.json`.
- `check_fresh.py` reruns and compares. It has been in the repository since the first commit, and nothing ran it.
- If the code breaks, the document should break. Here the number stopped matching and nothing said so.

```bash
cat metrics.json
python check_fresh.py
```

In [4]:
# Recorded output. You don't need to run this.
run("cat metrics.json", "python check_fresh.py")

$ cat metrics.json
{
  "growth_rate_ppm_per_year": 2.681,
  "rmse_ppm": 0.473,
  "seasonal_amplitude_ppm": 6.625
}

$ python check_fresh.py
STALE: metrics.json does not match a rerun of params.yaml and the data.
  growth_rate_ppm_per_year: metrics.json says 2.681, a rerun gives 2.749
  rmse_ppm: metrics.json says 0.473, a rerun gives 0.469
  seasonal_amplitude_ppm: metrics.json says 6.625, a rerun gives 6.63


- The tests pass anyway. They test the fit, not the committed number.

```bash
python -m pytest -q
```

In [5]:
# Recorded output. You don't need to run this.
run("python -m pytest -q")

$ python -m pytest -q
..                                                                       [100%]
2 passed in 0.02s


**Facts**

- One commit changed something without a rerun.
- `git bisect run` does a binary search through the history. It takes any command that exits 0 for good and nonzero for bad, and `check_fresh.py` is one.
- You could read six commits by hand. You couldn't read 600.
- A higher polynomial degree always fits the years it saw a little better, so that comparison can't choose the degree.
- `holdout.py` fits the years up to 2019 and scores each degree on 2020 onward, which the fit never saw.
- Choosing the degree is a decision. No tool makes it for you.

**Steps**

1. Find the commit. Note the commit ID it reports, and read that commit with `git show`.

```bash
git bisect start HEAD $(git rev-list --max-parents=0 HEAD)
git bisect run python check_fresh.py
```

2. End the search.

```bash
git bisect reset
```

3. Read the holdout table, and pick a degree you can defend.

```bash
python holdout.py
```

4. Set `degree` in `params.yaml`, then rerun, stage and confirm. `check_fresh.py` reads what you staged.

```bash
../envs/co2/bin/python analyze.py
git add params.yaml metrics.json
python check_fresh.py
```

5. Commit. In the message, name the commit `git bisect` found and say why you chose this degree.

```bash
git commit -m "REPLACE THIS with the commit ID bisect found, the degree you chose, and why"
```

6. Install the hook. It has been in the repository since the first commit, and nobody installed it.

```bash
cp hooks/pre-commit .git/hooks/pre-commit
```

7. Try to commit a stale number. Change `degree` in `params.yaml`, save it, and paste this. The hook should refuse the commit. The last two lines put `params.yaml` back.

```bash
git add params.yaml
git commit -m "Try to commit a stale number"
git restore --staged params.yaml
git restore params.yaml
```

## Step 3 (optional): delete everything that runs it

**Facts**

- The clone and the environment are cattle. If the project is right, you can delete them and get the same number back.
- Only what's on the remote survives, so push first.
- A clone doesn't carry hooks, so you install the hook again.
- A tag marks the release, so anyone can find this exact version later.

**Steps**

1. Run this. `git status --short` near the end should print nothing, which means you got the same number.

```bash
git push origin main
cd ..
rm -rf project envs/co2
git clone --config user.name="Workshop Learner" --config user.email="learner@example.org" remote/co2.git project
cd project
conda create --yes --quiet --prefix ../envs/co2 --file explicit-$(conda config --show subdir | cut -d' ' -f2).txt
../envs/co2/bin/python analyze.py
git status --short
git tag -a v1.0 -m "Publication version"
git push origin v1.0
```

2. Install the hook again in the new clone.

```bash
cp hooks/pre-commit .git/hooks/pre-commit
```

- On Binder, JupyterLab may say `Directory not found` while `project` is gone. Close the message. It does no harm, and you can open `project` again in the file panel.

## Check everything

- The full check, for steps 1 to 3. It reads what you committed and tells you what to do next.
- It knows the correct growth rate only as a salted digest, so reading `check.py` won't give you the number.

```bash
python ../check.py
```

On the starter project it fails like this:

In [6]:
# Recorded output. You don't need to run this.
run("python ../check.py")

$ python ../check.py
FAIL  analyze.py imports pandas, and the committed environment.yml does not declare it
FAIL  No lock file is committed. Write one with conda list --explicit --md5 and commit it.
FAIL  The committed metrics.json does not match a rerun. Run python check_fresh.py to see the numbers.
FAIL  A rerun of the committed params.yaml gives a growth rate this exercise doesn't accept. A rerun alone won't fix that. Compare the degrees with holdout.py, then decide.
FAIL  No commit message names the commit git bisect found, with at least its first 7 characters, and says why you chose this degree. Use the word degree and give your reason.
FAIL  The pre-commit hook is not installed in project/. Copy hooks/pre-commit into .git/hooks/. A clone does not bring hooks with it.
FAIL  No environment at envs/co2. Build it with the commands in step 1.
FAIL  remote/co2.git has no annotated tag v1.0. Use git tag -a v1.0, then git push origin v1.0.

0 of 8 parts pass.


## Summary

- `environment.yml` is what you asked for, and the lock is what you got. Commit both.
- A result in Git should match a rerun of what's in Git. A check like `check_fresh.py` tests that, and a hook enforces it.
- `git bisect run` finds the commit that changed a result.
- If you can delete the clone and the environment and get the same number back, they're cattle.

## Why this CSV isn't in Git LFS

- The CSV is 39 KB, and Git handles that fine.
- Large files make every clone heavy. Git LFS keeps them out of the repository, for example `git lfs track "*.pt"` for checkpoints.
- LFS needs a server. DVC, in exercise 2, doesn't.

## On your own project

- Check that the `metrics.json` or table you last shared matches what's in Git now.
- Add a check like `check_fresh.py`, and install it as a hook.
- Write the lock today.

## About the data

- Monthly mean CO2 at Mauna Loa, `data/co2_mm_mlo.csv`, 39 KB, unchanged from [NOAA Global Monitoring Laboratory](https://gml.noaa.gov/ccgg/trends/data.html).
- NOAA makes the data "freely available to the public and the scientific community" and asks for credit. The terms are in the file's header.
- Cite it as: Dr. Xin Lan, NOAA/GML (gml.noaa.gov/ccgg/trends/) and Dr. Ralph Keeling, Scripps Institution of Oceanography (scrippsco2.ucsd.edu/).